Split into dataframe: Activities, actualTime, quantity

In [2]:
import pandas as pd
import numpy as np
dcData = pd.read_csv('/Users/calebfruin/Downloads/job_cost.csv')
dcData['Employee'] = dcData['Employee'].str.split(' -').str[0]

dcData

import plotly.express as px

fig = px.bar(dcData, x='Employee', y='ActualTime', color= 'Activity Code', title='Actual Time by Employee and Department')
fig.show()

Getting workload by department: sum all entries by activities rolledUp by dept

In [3]:
import pandas as pd
df = pd.read_csv('5248_2026-Sept.csv')
#get duration in minutes from start and end times
df['start_dt'] = pd.to_datetime(df['START'], format='%m/%d/%Y %I:%M %p')
df['end_dt']   = pd.to_datetime(df['END'],   format='%m/%d/%Y %I:%M %p')
df['duration'] = df['end_dt'] - df['start_dt']
df['duration_minutes'] = df['duration'].dt.total_seconds() / 60
df['duration_hours'] = df['duration'].dt.total_seconds() / 3600

#Remove rows with negative values in the 'duration' column
df = df[df['duration'] >= pd.Timedelta(0)]

#rename activities to between "- " and " -"
df['Activity Code'] = df['Activity Code'].str.split(' -').str[1]

#df = df[df['Activity Code'] == '31520 - Canon VarioP 6330 -2 Run']
#monthly = monthly[monthly['Activity Code'].str.startswith('330')]
Activity = df.groupby(['Department', 'Activity Code'])['duration_hours'].sum().reset_index()


ActFig = px.bar(Activity, x='Department', y='duration_hours', color='Activity Code')
df[df['duration'].notnull()].head()
ActFig.show()
#Activity


FileNotFoundError: [Errno 2] No such file or directory: '5248_2026-Sept.csv'

Combine PACE.JOB/customer-datasets, extract KPI: Order-rate, fulfilment-time, plot job/month or week

In [19]:
import pandas as pd
import glob
file_paths = glob.glob('/Users/calebfruin/Downloads/*.xlsx')
JobDf = pd.concat((pd.read_excel(file) for file in file_paths), ignore_index=True)
#JobDf = pd.read_excel('Customer-Jobs/BYNUM (1).xlsx')
JobDf = JobDf[JobDf['Admin Status'].isin(['Closed'])] #filter out incomplete jobs
JobDf = JobDf[JobDf['Last Shipped'].isnull() == False] #filter out jobs with no ship date
JobDf = JobDf[pd.to_datetime(JobDf['Date Setup']) > pd.to_datetime(JobDf['Last Shipped'])] # Filter to Jobs supposed shipped before setup date
JobDf['Turnaround Time'] = (pd.to_datetime(JobDf['Last Shipped']) - pd.to_datetime(JobDf['Date Setup'])).dt.days
#JobDf['Turnaround Time'] = JobDf['Turnaround Time'].mean().astype(int) #.astype(str) + " days"
#JobDf[["Charge Back Account","Turnaround Time"]].groupby("Charge Back Account").mean()
JobDf[['Charge Back Account','Date Setup','Last Shipped']].head(25)

/Users/calebfruin/Library/CloudStorage/OneDrive-Personal/Documents/Projects/Shipments_Dashboarding/fedEx-Dashboard-Prototype/dashboard_env/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/calebfruin/Library/CloudStorage/OneDrive-Personal/Documents/Projects/Shipments_Dashboarding/fedEx-Dashboard-Prototype/dashboard_env/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/calebfruin/Library/CloudStorage/OneDrive-Personal/Documents/Projects/Shipments_Dashboarding/fedEx-Dashboard-Prototype/dashboard_env/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains n

,Charge Back Account,Date Setup,Last Shipped
74,IC990236,06/05/2026,12/31/1969
106,IC990236,03/13/2026,12/31/1969
108,IC990236,03/19/2026,12/31/1969
129,IC990236,12/11/2025,12/31/1969
171,IC990236,10/06/2025,12/31/1969
172,IC990236,10/06/2025,12/31/1969
6491,106676,08/05/2026,12/31/1969
6513,106676,05/07/2026,12/31/1969
6527,106676,03/16/2026,12/31/1969
6528,106676,03/13/2026,12/31/1969


In [ ]:
import sys
print(sys.executable)
print(sys.version)

/Users/calebfruin/Library/CloudStorage/OneDrive-Personal/Documents/Projects/Shipments_Dashboarding/fedEx-Dashboard-Prototype/dashboard_env/bin/python
3.12.10 (v3.12.10:0cc81280367, Apr  8 2025, 08:47:00) [Clang 13.0.0 (clang-1300.0.29.30)]


Knowing details in the dates for Status:Closed are broken for some groups, I will isolate them and review.